# DLAV Project - Phase 2

This notebook mirrors the `dlav_phase2.py` script. It includes data loading, model definition, training, visualization, and submission creation.

In [ ]:
# Mount Google Drive (optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception:
    DRIVE_ROOT = None
    print("Google Drive not available; saving locally.")

The first step is to prepare the data. The code below will download the data from Google Drive and extract it here for your code to use. Whenever your session gets restarted, remember to re-run this cell to re-download the data.

In [6]:
# Install gdown to handle Google Drive file download
!pip install -q gdown

import gdown
import zipfile

download_url = "https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr"
output_zip = "dlav_train.zip"
gdown.download(download_url, output_zip, quiet=False)
with zipfile.ZipFile(output_zip, "r") as zip_ref:
    zip_ref.extractall(".")

download_url = "https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu"
output_zip = "dlav_val.zip"
gdown.download(download_url, output_zip, quiet=False)
with zipfile.ZipFile(output_zip, "r") as zip_ref:
    zip_ref.extractall(".")

download_url = "https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV"
output_zip = "dlav_test_public.zip"
gdown.download(download_url, output_zip, quiet=False)
with zipfile.ZipFile(output_zip, "r") as zip_ref:
    zip_ref.extractall(".")

Downloading...
From (original): https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr
From (redirected): https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr&confirm=t&uuid=c34825e7-28c9-48f3-8477-e9a8b2d48f16
To: /home/paul/Documents/EPFL/Cours/DLAV/Project/dlav_train.zip
100%|██████████| 439M/439M [00:40<00:00, 10.9MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu
From (redirected): https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu&confirm=t&uuid=30a0ad16-e200-4a8e-9c2e-6b4a9b2b5c6b
To: /home/paul/Documents/EPFL/Cours/DLAV/Project/dlav_val.zip
100%|██████████| 87.8M/87.8M [00:07<00:00, 11.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV
From (redirected): https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV&confirm=t&uuid=8ba3f127-6a32-4f82-8804-73364bd1290c
To: /home/paul/Documents/EPFL/Cours/DLAV/Project/dlav_test_public.zip


In [ ]:
# Setup and configuration
import os
import copy
import random
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, update_bn

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

CFG = dict(
    # Data
    train_dir="train",
    val_dir="val",
    test_dir="test_public",
    num_workers=2,

    # Model
    degree_coarse=3,
    num_timesteps=60,
    hist_in_dim=7,         # x, y, heading + vx, vy + ax, ay
    hist_d_model=256,
    hist_nhead=4,
    hist_layers=4,
    cmd_embed_dim=64,
    d_model=256,           # shared token dim for spatial pool + decoder
    num_img_queries=8,     # learnable queries for spatial pooling
    decoder_nhead=4,
    decoder_layers=3,      # cross-attention decoder depth
    dropout=0.15,
    freeze_backbone=1,
    velocity_window=5,
    num_seg_classes=14,

    # Aux task flags
    use_depth=True,
    use_seg=True,

    # Mixed precision
    use_amp=True,

    # Loss weights
    lambda_coarse=0.2,
    lambda_depth=0.0,
    lambda_seg=0.0,
    traj_weight_end=1.0,   # PHASE 1: uniform weighting (ADE metric is uniform)
    aux_normalize=True,
    aux_target=0.05,

    # PHASE 1: speed augmentation — scale (x, y) of history+future by U(s_min, s_max)
    speed_aug=(0.8, 1.25),

    # PHASE 1: test-time augmentation — average pred(image) + pred(hflip image)
    use_tta=True,

    # Training
    batch_size=96,
    num_epochs=80,
    lr_backbone=2e-5,
    lr_head=7e-4,
    weight_decay=3e-4,

    # SWA
    swa_start_frac=0.75,

    # Checkpointing/resume
    checkpoint_last="checkpoint_last.pt",
    resume_from=None,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
class DrivingDataset(Dataset):
    """
    Loads each .pkl sample and returns:
      camera   : (3, H, W)  float32, ImageNet-normalized
      history  : (21, 7)    float32  [x, y, heading, vx, vy, ax, ay]
      command  : ()         int64    0=forward, 1=left, 2=right
      future   : (T, 3)     float32  [x, y, heading]  - absent for test
      depth    : (H, W, 1)  float32  - optional
      seg      : (H, W)     int64    - optional
    """
    COMMAND_TO_IDX = {"forward": 0, "left": 1, "right": 2}

    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    MAX_CROP_TOP = 30
    MAX_HSHIFT = 8

    def __init__(self, file_list, train=False, test=False, speed_aug=None):
        self.samples = file_list
        self.train = train
        self.test = test
        # PHASE 1: only apply speed aug during training
        self.speed_aug = speed_aug if train else None

        if train:
            self.color_aug = transforms.Compose([
                transforms.ColorJitter(brightness=0.25, contrast=0.25,
                                       saturation=0.15, hue=0.05),
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
                transforms.RandomErasing(p=0.25, scale=(0.02, 0.08),
                                         ratio=(0.3, 3.3), value=0),
            ])
        else:
            self.color_aug = None

    def _random_crop_top(self, img):
        crop = torch.randint(0, self.MAX_CROP_TOP + 1, (1,)).item()
        if crop == 0:
            return img
        cropped = img[:, crop:, :]
        return F.interpolate(cropped.unsqueeze(0), size=(200, 300),
                             mode="bilinear", align_corners=False).squeeze(0)

    def _random_hshift(self, img):
        shift = torch.randint(-self.MAX_HSHIFT, self.MAX_HSHIFT + 1, (1,)).item()
        if shift == 0:
            return img
        shifted = torch.roll(img, shifts=shift, dims=2)
        if shift > 0:
            shifted[:, :, :shift] = 0
        else:
            shifted[:, :, shift:] = 0
        return shifted

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        with open(self.samples[idx], "rb") as f:
            data = pickle.load(f)

        camera = torch.FloatTensor(data["camera"]).permute(2, 0, 1) / 255.0
        if self.train:
            camera = self._random_crop_top(camera)
            camera = self._random_hshift(camera)
            camera = self.color_aug(camera)
        camera = (camera - self.IMAGENET_MEAN) / self.IMAGENET_STD

        history_raw = torch.FloatTensor(data["sdc_history_feature"])  # (21, 3)
        future_raw = None
        if not self.test:
            future_raw = torch.FloatTensor(data["sdc_future_feature"])

        # PHASE 1: speed augmentation — scale positions by a random factor.
        # Velocities/accelerations are derived from positions below, so they
        # scale consistently. Heading (column 2) is left unchanged.
        if self.train and self.speed_aug is not None:
            s_min, s_max = self.speed_aug
            s = float(torch.empty(1).uniform_(s_min, s_max).item())
            history_raw = history_raw.clone()
            history_raw[:, :2] *= s
            if future_raw is not None:
                future_raw = future_raw.clone()
                future_raw[:, :2] *= s

        pos = history_raw[:, :2]
        vel = torch.zeros_like(pos); vel[1:] = pos[1:] - pos[:-1]
        acc = torch.zeros_like(pos); acc[1:] = vel[1:] - vel[:-1]
        history = torch.cat([history_raw, vel, acc], dim=-1)           # (21, 7)

        command = torch.tensor(
            self.COMMAND_TO_IDX[data["driving_command"]], dtype=torch.long
        )

        sample = {"camera": camera, "history": history, "command": command}

        if future_raw is not None:
            t = int(CFG["num_timesteps"])
            sample["future"] = future_raw[:t]

        if "depth" in data:
            sample["depth"] = torch.FloatTensor(data["depth"])
        if "semantic_label" in data:
            sample["seg"] = torch.LongTensor(data["semantic_label"].astype(np.int64))

        # Random horizontal flip: image L/R + negate y/heading/vy/ay in history & future,
        # and swap left/right driving command.
        if self.train and torch.rand(1).item() < 0.5:
            sample["camera"] = sample["camera"].flip(2)
            h = sample["history"].clone()
            h[:, 1] *= -1
            h[:, 2] *= -1
            h[:, 4] *= -1
            h[:, 6] *= -1
            sample["history"] = h
            if "future" in sample:
                f = sample["future"].clone()
                f[:, 1] *= -1
                sample["future"] = f
            if "depth" in sample:
                sample["depth"] = sample["depth"].flip(1)
            if "seg" in sample:
                sample["seg"] = sample["seg"].flip(1)

            cmd_int = int(sample["command"])
            if cmd_int == 1:
                sample["command"] = torch.tensor(2, dtype=torch.long)
            elif cmd_int == 2:
                sample["command"] = torch.tensor(1, dtype=torch.long)

        return sample

In [ ]:
class HistoryTokenEncoder(nn.Module):
    """
    Encodes (B, T_hist, in_dim) -> (B, 1+T_hist, d_model) — returns CLS token at
    index 0 plus per-step tokens, all in the shared d_model space so they can
    be used as memory for the cross-attention trajectory decoder.
    """
    def __init__(self, in_dim=7, d_model=256, nhead=4, num_layers=4,
                 dropout=0.1, max_len=22):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, history):
        b, t, _ = history.shape
        x = self.input_proj(history)
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1)                       # (B, 1+T, d)
        pos = torch.arange(x.size(1), device=history.device)
        x = x + self.pos_embed(pos).unsqueeze(0)
        x = self.transformer(x)
        return self.norm(x)                                  # (B, 1+T, d)


class SpatialAttentionPool(nn.Module):
    """
    PHASE 2: replaces GAP. K learnable queries cross-attend over the flattened
    spatial feature map. Returns both:
      * the K query-pooled tokens (for downstream summary / coarse head)
      * the projected spatial tokens (HxW long), used as memory for the
        trajectory decoder so it can attend to specific image regions.
    """
    def __init__(self, in_channels, d_model=256, num_queries=8, nhead=4,
                 dropout=0.1, max_hw=512):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=1)
        self.pos_embed = nn.Parameter(torch.zeros(max_hw, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.queries = nn.Parameter(torch.zeros(num_queries, d_model))
        nn.init.trunc_normal_(self.queries, std=0.02)

        self.norm_kv = nn.LayerNorm(d_model)
        self.norm_q = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True,
        )
        self.ffn = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )

    def forward(self, feat_map):
        b, _, h, w = feat_map.shape
        x = self.proj(feat_map)                                # (B, d, H, W)
        x = x.flatten(2).transpose(1, 2)                       # (B, H*W, d)
        n = x.size(1)
        x = x + self.pos_embed[:n].unsqueeze(0)
        x = self.norm_kv(x)

        q = self.queries.unsqueeze(0).expand(b, -1, -1)        # (B, K, d)
        qn = self.norm_q(q)
        attn_out, _ = self.attn(qn, x, x)                      # (B, K, d)
        pooled = q + attn_out
        pooled = pooled + self.ffn(pooled)
        return pooled, x                                        # (B, K, d), (B, H*W, d)


class PolynomialTrajectoryDecoder(nn.Module):
    """
    Kept from the previous architecture for two purposes:
      (1) supplying the linear-velocity prior that fine_traj is a residual on
      (2) the smooth-target coarse polynomial head used for regularization
    """
    def __init__(self, degree=5, num_timesteps=60, velocity_window=5):
        super().__init__()
        self.degree = degree
        self.num_timesteps = num_timesteps

        t = torch.linspace(0, 1, num_timesteps)
        v = torch.stack([t ** (k + 1) for k in range(degree)], dim=1)
        self.register_buffer("V", v)

        t_lin = torch.arange(1, num_timesteps + 1, dtype=torch.float32)
        self.register_buffer("t_linear", t_lin)

    def compute_prior(self, history):
        # 3-step average velocity for stability
        vel = (history[:, -1, :2] - history[:, -4, :2]) / 3
        return self.t_linear.view(1, -1, 1) * vel.unsqueeze(1)

    def compute_residual(self, coeffs):
        return torch.einsum("tk,bak->bta", self.V, coeffs)

    def forward(self, coeffs, history, add_prior=True):
        residual = self.compute_residual(coeffs)
        if not add_prior:
            return residual
        prior = self.compute_prior(history)
        return prior + residual


class TrajectoryCrossAttnDecoder(nn.Module):
    """
    PHASE 2: replaces the polynomial fine decoder. T learnable timestep queries
    (with learned temporal embeddings) cross-attend to a memory composed of:
      * spatial image tokens (per-region attention)
      * history tokens (incl. CLS)
      * command token
    Each query produces a 2-D residual added on top of the linear-velocity
    prior outside this module.
    """
    def __init__(self, d_model=256, nhead=4, num_layers=3,
                 num_timesteps=60, dropout=0.1):
        super().__init__()
        self.T = num_timesteps
        self.query_embed = nn.Parameter(torch.zeros(num_timesteps, d_model))
        nn.init.trunc_normal_(self.query_embed, std=0.02)
        self.time_embed = nn.Embedding(num_timesteps, d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 2)
        # Zero-init so training starts at prior
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, memory):
        b = memory.size(0)
        t_idx = torch.arange(self.T, device=memory.device)
        q = self.query_embed.unsqueeze(0) + self.time_embed(t_idx).unsqueeze(0)
        q = q.expand(b, -1, -1)                                  # (B, T, d)
        out = self.decoder(q, memory)                            # (B, T, d)
        return self.head(self.norm(out))                         # (B, T, 2)


def build_aux_decoder(in_channels, out_channels, target_size=(200, 300)):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, 256, 4, stride=2, padding=1), nn.ReLU(inplace=True),
        nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.ReLU(inplace=True),
        nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
        nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(32, out_channels, 3, padding=1),
        nn.Upsample(size=target_size, mode="bilinear", align_corners=False),
    )

In [ ]:
class DrivingPlannerV2(nn.Module):
    """
    PHASE 2 architecture:
      - ConvNeXt-Tiny backbone (bigger / more modern than ResNet-34)
      - SpatialAttentionPool replaces GAP (preserves spatial information)
      - TrajectoryCrossAttnDecoder replaces the polynomial fine head
      - Linear-velocity prior preserved; fine_traj = prior + residual
      - Coarse poly head kept as a smooth-target regularizer
    """
    def __init__(self, cfg):
        super().__init__()
        self.use_depth = cfg["use_depth"]
        self.use_seg = cfg["use_seg"]
        d_model = cfg["d_model"]
        self.d_model = d_model
        self.T = cfg["num_timesteps"]

        # ── ConvNeXt-Tiny backbone, split into 4 stages ───────────────────────
        cn = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        feats = cn.features  # 8 modules: stem, blk1, down1, blk2, down2, blk3, down3, blk4
        self.stage0 = nn.Sequential(feats[0], feats[1])  # /4,  96  ch
        self.stage1 = nn.Sequential(feats[2], feats[3])  # /8,  192 ch
        self.stage2 = nn.Sequential(feats[4], feats[5])  # /16, 384 ch
        self.stage3 = nn.Sequential(feats[6], feats[7])  # /32, 768 ch
        self.backbone_out_channels = 768
        self._freeze_stages(cfg["freeze_backbone"])

        # ── Spatial attention pool over /32 feature map ───────────────────────
        self.spatial_pool = SpatialAttentionPool(
            in_channels=self.backbone_out_channels,
            d_model=d_model,
            num_queries=cfg["num_img_queries"],
            nhead=cfg["decoder_nhead"],
            dropout=cfg["dropout"],
        )

        # ── History encoder: returns CLS + per-step tokens in d_model ────────
        self.history_encoder = HistoryTokenEncoder(
            in_dim=cfg["hist_in_dim"],
            d_model=d_model,
            nhead=cfg["hist_nhead"],
            num_layers=cfg["hist_layers"],
            dropout=cfg["dropout"],
            max_len=22,
        )

        # ── Command embedding → single token of d_model ──────────────────────
        self.command_embedding = nn.Embedding(3, d_model)

        # ── Coarse poly head (kept for smooth-target regularization) ─────────
        self.summary_proj = nn.Linear(d_model, d_model)
        self.coarse_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(d_model // 2, 2 * cfg["degree_coarse"]),
        )
        nn.init.zeros_(self.coarse_head[-1].weight)
        nn.init.zeros_(self.coarse_head[-1].bias)
        self.poly_decoder = PolynomialTrajectoryDecoder(
            degree=cfg["degree_coarse"],
            num_timesteps=cfg["num_timesteps"],
            velocity_window=cfg["velocity_window"],
        )

        # ── Cross-attention trajectory decoder ───────────────────────────────
        self.traj_decoder = TrajectoryCrossAttnDecoder(
            d_model=d_model,
            nhead=cfg["decoder_nhead"],
            num_layers=cfg["decoder_layers"],
            num_timesteps=cfg["num_timesteps"],
            dropout=cfg["dropout"],
        )

        # ── Aux heads (from /32 feature map) ─────────────────────────────────
        if self.use_depth:
            self.depth_decoder = build_aux_decoder(self.backbone_out_channels, 1)
        if self.use_seg:
            self.seg_decoder = build_aux_decoder(
                self.backbone_out_channels, cfg["num_seg_classes"],
            )

    def _freeze_stages(self, n):
        stages = [self.stage0, self.stage1, self.stage2, self.stage3]
        n = max(0, min(int(n), len(stages)))
        for i, stage in enumerate(stages):
            if i < n:
                for p in stage.parameters():
                    p.requires_grad = False

    def get_param_groups(self, lr_backbone, lr_head, weight_decay):
        backbone_params = [
            p for stage in [self.stage0, self.stage1, self.stage2, self.stage3]
            for p in stage.parameters() if p.requires_grad
        ]
        head_modules = [
            self.spatial_pool, self.history_encoder,
            self.command_embedding,
            self.summary_proj, self.coarse_head, self.poly_decoder,
            self.traj_decoder,
        ]
        if self.use_depth:
            head_modules.append(self.depth_decoder)
        if self.use_seg:
            head_modules.append(self.seg_decoder)
        head_params = [p for m in head_modules for p in m.parameters()]
        return [
            {"params": backbone_params, "lr": lr_backbone, "weight_decay": weight_decay},
            {"params": head_params, "lr": lr_head, "weight_decay": weight_decay},
        ]

    def forward(self, camera, history, command):
        # Backbone feature pyramid
        f0 = self.stage0(camera)
        f1 = self.stage1(f0)
        f2 = self.stage2(f1)
        f3 = self.stage3(f2)

        # Spatial attention pool over deepest feature map
        img_queries, img_tokens = self.spatial_pool(f3)        # (B, K, d), (B, HW, d)

        # History tokens (CLS at index 0)
        hist_tokens = self.history_encoder(history)            # (B, 1+T_hist, d)

        # Command token
        cmd_tok = self.command_embedding(command).unsqueeze(1)  # (B, 1, d)

        # Memory for cross-attention decoder: image regions + history + command
        memory = torch.cat([img_tokens, hist_tokens, cmd_tok], dim=1)

        # Coarse polynomial head, conditioned on hist CLS + pooled image + cmd
        summary = (
            hist_tokens[:, 0]
            + img_queries.mean(dim=1)
            + cmd_tok.squeeze(1)
        )
        latent = self.summary_proj(summary)
        coarse_coeffs = self.coarse_head(latent).view(
            -1, 2, self.poly_decoder.degree,
        )
        coarse_traj = self.poly_decoder(coarse_coeffs, history)  # (B, T, 2)

        # Fine trajectory: prior + cross-attention residual
        prior = self.poly_decoder.compute_prior(history)         # (B, T, 2)
        fine_residual = self.traj_decoder(memory)                # (B, T, 2)
        fine_traj = prior + fine_residual

        depth_out = torch.sigmoid(self.depth_decoder(f3)) if self.use_depth else None
        seg_out = self.seg_decoder(f3) if self.use_seg else None

        return fine_traj, coarse_traj, depth_out, seg_out

In [ ]:
def smooth_trajectory(traj, degree=3, num_timesteps=60):
    """Polynomial fit on CPU in float32 — avoids AMP autocast casting matmuls to fp16."""
    orig_dtype, orig_device = traj.dtype, traj.device
    traj_cpu = traj.detach().cpu().float()          # (b, t, c) on CPU, float32
    b, t, c = traj_cpu.shape
    x = torch.linspace(0, 1, t, dtype=torch.float32)
    V = torch.stack([x ** k for k in range(degree + 1)], dim=1)  # (t, d+1)
    V_pinv = torch.linalg.pinv(V)                                  # (d+1, t)
    coeffs = torch.einsum("dt,btc->bdc", V_pinv, traj_cpu)        # (b, d+1, c)
    smooth = torch.einsum("td,bdc->btc", V, coeffs)               # (b, t,   c)
    return smooth.to(dtype=orig_dtype, device=orig_device).detach()


def compute_loss(fine_traj, coarse_traj, depth_out, seg_out,
                 batch, cfg, device):
    gt_future = batch["future"].to(device)[..., :2]

    t = gt_future.size(1)
    w_end = float(cfg.get("traj_weight_end", 2.0))
    weights = torch.linspace(1.0, w_end, t, device=gt_future.device)
    weights = weights / weights.mean()
    diff = fine_traj - gt_future
    # Stable L2 norm: avoids undefined gradient at exactly zero displacement
    dist = (diff.pow(2).sum(dim=-1) + 1e-8).sqrt()
    traj_loss = (dist * weights).mean()

    smooth_gt = smooth_trajectory(gt_future, degree=cfg["degree_coarse"])
    coarse_loss = F.mse_loss(coarse_traj, smooth_gt)

    base_loss = traj_loss + cfg["lambda_coarse"] * coarse_loss
    total = base_loss
    log = {"traj": traj_loss.item(), "coarse": coarse_loss.item()}

    aux_normalize = bool(cfg.get("aux_normalize", False))
    aux_target = float(cfg.get("aux_target", 0.1))
    eps = 1e-6

    if depth_out is not None and "depth" in batch:
        gt_depth = batch["depth"].to(device)
        gt_depth = gt_depth.permute(0, 3, 1, 2)
        d_max = gt_depth.flatten(1).max(1)[0].view(-1, 1, 1, 1).clamp(min=1e-3)
        gt_depth = gt_depth / d_max
        depth_loss = F.l1_loss(depth_out, gt_depth)
        if aux_normalize:
            depth_scaled = aux_target * base_loss.detach() * (depth_loss / (depth_loss.detach() + eps))
        else:
            depth_scaled = cfg["lambda_depth"] * depth_loss
        total = total + depth_scaled
        log["depth"] = depth_loss.item()
        log["depth_scaled"] = depth_scaled.item()

    if seg_out is not None and "seg" in batch:
        gt_seg = batch["seg"].to(device)
        seg_loss = F.cross_entropy(seg_out, gt_seg, ignore_index=255)
        if aux_normalize:
            seg_scaled = aux_target * base_loss.detach() * (seg_loss / (seg_loss.detach() + eps))
        else:
            seg_scaled = cfg["lambda_seg"] * seg_loss
        total = total + seg_scaled
        log["seg"] = seg_loss.item()
        log["seg_scaled"] = seg_scaled.item()

    log["total"] = total.item()
    return total, log


def compute_ade_fde(pred, gt):
    diff = pred - gt[..., :2]
    dist = torch.norm(diff, p=2, dim=-1)
    ade = dist.mean(dim=1).mean().item()
    fde = dist[:, -1].mean().item()
    return ade, fde

In [ ]:
def tta_forward(model, camera, history, command):
    """
    PHASE 1: test-time augmentation. Predict on the image and its horizontal
    flip (with history reflected and left/right commands swapped), then
    average after un-flipping the prediction's y-coordinate. Returns
    fine_traj only — aux outputs aren't needed during evaluation.
    """
    fine1, *_ = model(camera, history, command)

    camera_f = camera.flip(-1)
    history_f = history.clone()
    history_f[..., 1] *= -1   # y
    history_f[..., 2] *= -1   # heading
    history_f[..., 4] *= -1   # vy
    history_f[..., 6] *= -1   # ay
    cmd_f = torch.where(
        command == 1, torch.full_like(command, 2),
        torch.where(command == 2, torch.full_like(command, 1), command),
    )

    fine2, *_ = model(camera_f, history_f, cmd_f)
    fine2 = fine2.clone()
    fine2[..., 1] *= -1

    return (fine1 + fine2) / 2


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_ade, history_log):
    state = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "best_ade": best_ade,
        "history_log": history_log,
    }
    torch.save(state, path)


def _load_state_dict_compatible(model, state_dict):
    model_state = model.state_dict()
    filtered = {}
    skipped = []
    for k, v in state_dict.items():
        if k in model_state and model_state[k].shape == v.shape:
            filtered[k] = v
        else:
            skipped.append(k)
    missing, unexpected = model.load_state_dict(filtered, strict=False)
    return filtered, skipped, missing, unexpected


def load_checkpoint(path, model, optimizer=None, scheduler=None, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and "model" in ckpt:
        _, skipped, _, _ = _load_state_dict_compatible(model, ckpt["model"])
        if skipped:
            print(f"Skipped {len(skipped)} incompatible keys from checkpoint.")
        if optimizer is not None and ckpt.get("optimizer") is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt.get("scheduler") is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = int(ckpt.get("epoch", 0))
        best_ade = float(ckpt.get("best_ade", float("inf")))
        history_log = ckpt.get(
            "history_log",
            {"train_total": [], "val_ade": [], "val_fde": [], "lr": []},
        )
        return start_epoch, best_ade, history_log

    _, skipped, _, _ = _load_state_dict_compatible(model, ckpt)
    if skipped:
        print(f"Skipped {len(skipped)} incompatible keys from checkpoint.")
    history_log = {"train_total": [], "val_ade": [], "val_fde": [], "lr": []}
    return 0, float("inf"), history_log


def train(
    model, train_loader, val_loader, optimizer, scheduler, cfg,
    start_epoch=0, best_ade=float("inf"), history_log=None,
    checkpoint_path=None, save_every=1, print_every=50,
    swa_model=None, swa_start=None,
):
    import contextlib

    model = model.to(DEVICE)

    use_amp = bool(cfg.get("use_amp", False)) and (DEVICE.type == "cuda")
    use_tta = bool(cfg.get("use_tta", False))
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    autocast_ctx = torch.cuda.amp.autocast if DEVICE.type == "cuda" else contextlib.nullcontext

    best_state_dict = None
    if history_log is None:
        history_log = {"train_total": [], "val_ade": [], "val_fde": [], "lr": []}

    total_epochs = start_epoch + cfg["num_epochs"]
    for epoch in range(start_epoch, total_epochs):
        model.train()
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            camera = batch["camera"].to(DEVICE)
            history = batch["history"].to(DEVICE)
            command = batch["command"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with autocast_ctx(enabled=use_amp):
                fine_traj, coarse_traj, depth_out, seg_out = model(camera, history, command)
                loss, log = compute_loss(
                    fine_traj, coarse_traj, depth_out, seg_out, batch, cfg, DEVICE,
                )

            if print_every and (step == 0 or ((step + 1) % int(print_every) == 0)):
                lam_coarse = float(cfg.get("lambda_coarse", 0.0))
                msg = (
                    f"  [train] epoch {epoch + 1}/{total_epochs} "
                    f"batch {step + 1}/{len(train_loader)} | "
                    f"traj={log['traj']:.4f} "
                    f"+ coarse={log['coarse']:.4f} (lam={lam_coarse:.3g})"
                )
                if "depth_scaled" in log:
                    msg += f" + depth={log['depth']:.4f} (scaled={log['depth_scaled']:.4f})"
                if "seg_scaled" in log:
                    msg += f" + seg={log['seg']:.4f} (scaled={log['seg_scaled']:.4f})"
                msg += f" => total={log['total']:.4f}"
                print(msg)

            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)

        # SWA: accumulate averaged weights in the last fraction of training
        if swa_model is not None and swa_start is not None and epoch >= swa_start:
            swa_model.update_parameters(model)

        model.eval()
        ade_all, fde_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                camera = batch["camera"].to(DEVICE)
                history = batch["history"].to(DEVICE)
                command = batch["command"].to(DEVICE)
                future = batch["future"].to(DEVICE)

                with autocast_ctx(enabled=use_amp):
                    if use_tta:
                        fine_traj = tta_forward(model, camera, history, command)
                    else:
                        fine_traj, *_ = model(camera, history, command)
                ade, fde = compute_ade_fde(fine_traj, future)
                ade_all.append(ade)
                fde_all.append(fde)

        mean_ade = float(np.mean(ade_all))
        mean_fde = float(np.mean(fde_all))

        current_lr = optimizer.param_groups[0]["lr"]
        if scheduler is not None:
            scheduler.step()

        history_log["train_total"].append(epoch_loss)
        history_log["val_ade"].append(mean_ade)
        history_log["val_fde"].append(mean_fde)
        history_log["lr"].append(current_lr)

        is_best = mean_ade < best_ade
        if is_best:
            best_ade = mean_ade
            best_state_dict = copy.deepcopy(model.state_dict())

        if checkpoint_path and ((epoch + 1) % save_every == 0):
            save_checkpoint(
                checkpoint_path, model, optimizer, scheduler,
                epoch + 1, best_ade, history_log,
            )

        swa_active = swa_model is not None and swa_start is not None and epoch >= swa_start
        marker = " * (SWA active)" if (is_best and swa_active) else (" *" if is_best else "")
        print(
            f"Epoch {epoch + 1:3d}/{total_epochs} | "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Val ADE: {mean_ade:.4f} | FDE: {mean_fde:.4f} | "
            f"LR: {current_lr:.1e}{marker}"
        )

    print(f"Best ADE (base model): {best_ade:.4f}")
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    return model, history_log, best_ade

In [ ]:
IMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
IMAGENET_STD_NP = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)


def denorm(img_tensor):
    img = img_tensor.numpy() * IMAGENET_STD_NP + IMAGENET_MEAN_NP
    return np.clip(img.transpose(1, 2, 0), 0, 1)


def visualize(model, val_loader, k=4):
    model.eval()
    batch = next(iter(val_loader))
    camera = batch["camera"].to(DEVICE)
    history = batch["history"].to(DEVICE)
    command = batch["command"].to(DEVICE)
    future = batch["future"]

    with torch.no_grad():
        fine_traj, coarse_traj, depth_out, seg_out = model(camera, history, command)

    cam_np = camera.cpu()
    hist_np = history.cpu().numpy()
    fut_np = future.numpy()
    fine_np = fine_traj.cpu().numpy()
    coarse_np = coarse_traj.cpu().numpy()

    indices = random.choices(range(len(cam_np)), k=k)

    fig, axes = plt.subplots(1, k, figsize=(4 * k, 4))
    for i, idx in enumerate(indices):
        axes[i].imshow(denorm(cam_np[idx]))
        axes[i].axis("off")
        axes[i].set_title(f"Example {i + 1}")
    plt.suptitle("Camera Inputs")
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, k, figsize=(4 * k, 4))
    for i, idx in enumerate(indices):
        axes[i].plot(hist_np[idx, :, 0], hist_np[idx, :, 1],
                     "o-", color="gold", markersize=4, label="Past")
        axes[i].plot(fut_np[idx, :, 0], fut_np[idx, :, 1],
                     "o-", color="green", markersize=4, label="GT")
        axes[i].plot(coarse_np[idx, :, 0], coarse_np[idx, :, 1],
                     "--", color="orange", markersize=3, label="Coarse")
        axes[i].plot(fine_np[idx, :, 0], fine_np[idx, :, 1],
                     "o-", color="red", markersize=3, label="Fine")
        axes[i].legend(fontsize=7)
        axes[i].axis("equal")
        axes[i].grid(alpha=0.3)
    plt.suptitle("Trajectory: GT vs Coarse vs Fine")
    plt.tight_layout()
    plt.show()

    if depth_out is not None and "depth" in batch:
        dep_pred_np = depth_out.cpu().numpy()
        dep_gt_np = batch["depth"].numpy()
        fig, axes = plt.subplots(2, k, figsize=(4 * k, 6))
        for i, idx in enumerate(indices):
            axes[0, i].imshow(dep_gt_np[idx, :, :, 0], cmap="viridis")
            axes[0, i].set_title("GT Depth")
            axes[0, i].axis("off")
            axes[1, i].imshow(dep_pred_np[idx, 0, :, :], cmap="viridis")
            axes[1, i].set_title("Pred Depth")
            axes[1, i].axis("off")
        plt.suptitle("Depth Estimation")
        plt.tight_layout()
        plt.show()

    if seg_out is not None and "seg" in batch:
        seg_pred_np = seg_out.argmax(1).cpu().numpy()
        seg_gt_np = batch["seg"].numpy()
        fig, axes = plt.subplots(2, k, figsize=(4 * k, 6))
        for i, idx in enumerate(indices):
            axes[0, i].imshow(seg_gt_np[idx], cmap="tab20")
            axes[0, i].set_title("GT Seg")
            axes[0, i].axis("off")
            axes[1, i].imshow(seg_pred_np[idx], cmap="tab20")
            axes[1, i].set_title("Pred Seg")
            axes[1, i].axis("off")
        plt.suptitle("Segmentation")
        plt.tight_layout()
        plt.show()

In [ ]:
def make_submission(model, cfg, filename="submission_phase2.csv"):
    test_dir = cfg["test_dir"]
    test_files = [
        os.path.join(test_dir, fn)
        for fn in sorted(
            [f for f in os.listdir(test_dir) if f.endswith(".pkl")],
            key=lambda fn: int(os.path.splitext(fn)[0]),
        )
    ]
    test_dataset = DrivingDataset(test_files, train=False, test=True)
    test_loader = DataLoader(test_dataset, batch_size=250,
                             num_workers=cfg["num_workers"])

    use_tta = bool(cfg.get("use_tta", False))

    model.eval()
    all_plans = []
    with torch.no_grad():
        for batch in test_loader:
            camera = batch["camera"].to(DEVICE)
            history = batch["history"].to(DEVICE)
            command = batch["command"].to(DEVICE)
            if use_tta:
                fine_traj = tta_forward(model, camera, history, command)
            else:
                fine_traj, *_ = model(camera, history, command)
            all_plans.append(fine_traj.cpu().numpy()[..., :2])

    all_plans = np.concatenate(all_plans, axis=0)
    n, t, _ = all_plans.shape
    flat = all_plans.reshape(n, t * 2)

    cols = ["id"] + [f"{ax}_{t + 1}" for t in range(t) for ax in ("x", "y")]
    df = pd.DataFrame(
        np.hstack([np.arange(n).reshape(-1, 1), flat]),
        columns=cols,
    )
    df["id"] = df["id"].astype(int)
    df.to_csv(filename, index=False)
    print(f"Saved {filename} ({n} samples, {t} timesteps)")
    return df

In [ ]:
def main():
    train_files = [
        os.path.join(CFG["train_dir"], f)
        for f in os.listdir(CFG["train_dir"]) if f.endswith(".pkl")
    ]
    val_files = [
        os.path.join(CFG["val_dir"], f)
        for f in os.listdir(CFG["val_dir"]) if f.endswith(".pkl")
    ]
    print(f"Train: {len(train_files)} samples | Val: {len(val_files)} samples")

    # PHASE 1: pass speed_aug into the train dataset
    train_dataset = DrivingDataset(
        train_files, train=True, speed_aug=CFG.get("speed_aug"),
    )
    val_dataset = DrivingDataset(val_files, train=False)

    train_loader = DataLoader(
        train_dataset, batch_size=CFG["batch_size"],
        num_workers=CFG["num_workers"], shuffle=True,
        pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=CFG["batch_size"],
        num_workers=CFG["num_workers"], shuffle=False,
        pin_memory=True,
    )

    model = DrivingPlannerV2(CFG)
    n_total = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params total: {n_total:,} | Trainable: {n_trainable:,}")

    param_groups = model.get_param_groups(
        lr_backbone=CFG["lr_backbone"],
        lr_head=CFG["lr_head"],
        weight_decay=CFG["weight_decay"],
    )
    optimizer = optim.AdamW(param_groups)

    # PHASE 1: plain CosineAnnealingLR — no restarts (avoids dumping learned features)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["num_epochs"], eta_min=1e-6,
    )

    # SWA: start averaging at swa_start_frac of total training
    swa_model = AveragedModel(model)
    swa_start = int(CFG["swa_start_frac"] * CFG["num_epochs"])
    print(f"SWA will start at epoch {swa_start}/{CFG['num_epochs']}")

    start_epoch = 0
    best_ade = float("inf")
    history_log = None
    resume_from = CFG.get("resume_from")
    if resume_from:
        start_epoch, best_ade, history_log = load_checkpoint(
            resume_from, model, optimizer, scheduler, device=DEVICE,
        )
        print(f"Resumed from {resume_from} at epoch {start_epoch} (best ADE {best_ade:.4f})")

    model, history_log, best_ade = train(
        model, train_loader, val_loader, optimizer, scheduler, CFG,
        start_epoch=start_epoch, best_ade=best_ade, history_log=history_log,
        checkpoint_path=CFG.get("checkpoint_last"),
        swa_model=swa_model, swa_start=swa_start,
    )

    # ── SWA finalisation ────────────────────────────────────────────────────
    print("Updating BatchNorm statistics for SWA model...")
    swa_model = swa_model.to(DEVICE)
    update_bn(train_loader, swa_model, device=DEVICE)

    use_tta = bool(CFG.get("use_tta", False))
    swa_model.eval()
    ade_all, fde_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            camera  = batch["camera"].to(DEVICE)
            history = batch["history"].to(DEVICE)
            command = batch["command"].to(DEVICE)
            future  = batch["future"].to(DEVICE)
            if use_tta:
                fine_traj = tta_forward(swa_model, camera, history, command)
            else:
                fine_traj, *_ = swa_model(camera, history, command)
            ade, fde = compute_ade_fde(fine_traj, future)
            ade_all.append(ade); fde_all.append(fde)
    swa_ade = float(np.mean(ade_all))
    swa_fde = float(np.mean(fde_all))
    print(f"SWA model  — Val ADE: {swa_ade:.4f} | FDE: {swa_fde:.4f}")
    print(f"Base model — Val ADE: {best_ade:.4f}")

    final_model = swa_model if swa_ade < best_ade else model
    final_ade   = min(swa_ade, best_ade)
    print(f"Using {'SWA' if swa_ade < best_ade else 'base'} model for submission (ADE {final_ade:.4f})")

    # ── Plots ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history_log["train_total"], label="Train Loss")
    axes[0].set_title("Train Loss"); axes[0].grid(alpha=0.3)
    axes[1].plot(history_log["val_ade"], label="Val ADE")
    axes[1].axvline(swa_start, color="orange", linestyle="--", label="SWA start")
    axes[1].set_title("Val ADE"); axes[1].grid(alpha=0.3)
    axes[2].plot(history_log["val_fde"], label="Val FDE")
    axes[2].set_title("Val FDE"); axes[2].grid(alpha=0.3)
    for ax in axes: ax.legend()
    plt.tight_layout(); plt.show()

    visualize(final_model, val_loader, k=4)

    submission_name = "submission_phase2.csv"
    make_submission(final_model, CFG, filename=submission_name)

    checkpoint_name = "best_model_phase2.pt"
    # Save the raw state dict (unwrap AveragedModel if needed)
    raw_state = final_model.module.state_dict() if hasattr(final_model, "module") else final_model.state_dict()
    torch.save(raw_state, checkpoint_name)
    print(f"Checkpoint saved: {checkpoint_name}")

    if DRIVE_ROOT:
        drive_submission = os.path.join(DRIVE_ROOT, submission_name)
        drive_checkpoint  = os.path.join(DRIVE_ROOT, checkpoint_name)
        if os.path.isdir(DRIVE_ROOT):
            import shutil
            shutil.copy2(submission_name, drive_submission)
            shutil.copy2(checkpoint_name, drive_checkpoint)
            print(f"Copied to Drive: {drive_submission}")

    return final_model, history_log


# Run the full pipeline
main()